#**Pre-Trained BERT Model**






We can download this model from Keras Hub. We will work with a subsequently released version of BERT, RoBERTA, which is robustly optimized (Ro). The model's main value was that it was trained on 10x the amount of data though, so it's better tuned.

In [1]:
import keras_hub

# Backbone here refers to the RoBERTa base layers
tokenizer = keras_hub.models.Tokenizer.from_preset("roberta_base_en")
backbone = keras_hub.models.Backbone.from_preset("roberta_base_en")

We need to use the tokenizer that goes with the model to make sure it pre-processes our text in a way that the model expects.

In [2]:
tokenizer("The quick brown fox")

<tf.Tensor: shape=(4,), dtype=int32, numpy=array([  133,  2119,  6219, 23602], dtype=int32)>

Here are the base layers we loaded. Notice that it is essentially a token embedding layer with some normalization afterward, some dropout to the embeddings to avoid overfitting, and then a ton of stacked transformer encoders.

In [3]:
backbone.summary()

Model: "roberta_backbone"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ token_ids           │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embeddings          │ (None, None, 768) │ 38,996,736 │ token_ids[0][0]   │
│ (TokenAndPositionE… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embeddings_layer_n… │ (None, None, 768) │      1,536 │ embeddings[0][0]  │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embeddings_dropout  │ (None, None, 768) │          0 │ embeddings_layer… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ padding_mask        │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_0 │ (None, None, 768) │  7,087,872 │ embeddings_dropo… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_1 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_2 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_3 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_4 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_5 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_6 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_7 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_8 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_9 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_… │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_… │ (None, None, 768) │  7,087,872 │ transformer_laye

 Total params: 124,052,736 (473.22 MB)

 Trainable params: 124,052,736 (473.22 MB)

 Non-trainable params: 0 (0.00 B)

Let's try using this pre-trained model with our IMDB reviews...

In [4]:
import os, pathlib, shutil, random, keras

zip_path = keras.utils.get_file(
    origin="https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz",
    fname="imdb",
    extract=True,
)

imdb_extract_dir = pathlib.Path(zip_path) / "aclImdb"
train_dir = pathlib.Path("imdb_train")
test_dir = pathlib.Path("imdb_test")
val_dir = pathlib.Path("imdb_val")

shutil.copytree(imdb_extract_dir / "test", test_dir, dirs_exist_ok=True)

val_percentage = 0.2
for category in ("neg", "pos"):
    src_dir = imdb_extract_dir / "train" / category
    src_files = os.listdir(src_dir)
    random.Random(1337).shuffle(src_files)
    num_val_samples = int(len(src_files) * val_percentage)

    os.makedirs(train_dir / category, exist_ok=True)
    os.makedirs(val_dir / category, exist_ok=True)
    for index, file in enumerate(src_files):
        if index < num_val_samples:
            shutil.copy(src_dir / file, val_dir / category / file)
        else:
            shutil.copy(src_dir / file, train_dir / category / file)

84125825/84125825 ━━━━━━━━━━━━━━━━━━━━ 13s 0us/step


Create our Tensorflow Dataset objects from the .txt files now

In [5]:
batch_size = 16
train_ds = keras.utils.text_dataset_from_directory(
    train_dir, batch_size=batch_size
)
val_ds = keras.utils.text_dataset_from_directory(
    val_dir, batch_size=batch_size
)
test_ds = keras.utils.text_dataset_from_directory(
    test_dir, batch_size=batch_size
)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


We apply the RoBERTa tokenizer to our datasets now. Before we go to the RoBERTa tokenizer, however, we need to add some start and end tokens, and some padding tokens, that were present in the dataset that was used for training RoBERTa.

In [6]:
# We are 'packing' on some additional tokens here. Sequences up to 512 tokens, adding the start token to our reviews, the end token, and the padding token for reviews shorter than 512 words.
def preprocess(text, label):
    packer = keras_hub.layers.StartEndPacker(
        sequence_length=512,
        start_value=tokenizer.start_token_id,
        end_value=tokenizer.end_token_id,
        pad_value=tokenizer.pad_token_id,
        return_padding_mask=True,
    )

    # After adding those tokens, we can apply RoBERTa's tokenizer
    token_ids, padding_mask = packer(tokenizer(text))
    return {"token_ids": token_ids, "padding_mask": padding_mask}, label

# And now that our preprocessing function is written, we can apply it to our Tensorflow Datasets
preprocessed_train_ds = train_ds.map(preprocess)
preprocessed_val_ds = val_ds.map(preprocess)
preprocessed_test_ds = test_ds.map(preprocess)

Here is a pre-processed batch of data. We have integer sequences per review, and a masking vector for each one that tells RoBERTa which tokens it can ignore at inference.

In [7]:
next(iter(preprocessed_train_ds))

({'token_ids': <tf.Tensor: shape=(16, 512), dtype=int32, numpy=
  array([[   0,  713,   16, ...,    1,    1,    1],
         [   0,  133,   80, ...,    1,    1,    1],
         [   0, 1121,   10, ...,    1,    1,    1],
         ...,
         [   0, 2709,   10, ...,    1,    1,    1],
         [   0,  713,  371, ...,    1,    1,    1],
         [   0,  133,  181, ...,    1,    1,    1]], dtype=int32)>,
  'padding_mask': <tf.Tensor: shape=(16, 512), dtype=bool, numpy=
  array([[ True,  True,  True, ..., False, False, False],
         [ True,  True,  True, ..., False, False, False],
         [ True,  True,  True, ..., False, False, False],
         ...,
         [ True,  True,  True, ..., False, False, False],
         [ True,  True,  True, ..., False, False, False],
         [ True,  True,  True, ..., False, False, False]])>},
 <tf.Tensor: shape=(16,), dtype=int32, numpy=array([1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0], dtype=int32)>)

Okay, now we can add some Dense layers onto the RoBERTa backbone and use it for our predictions!

In [9]:
from keras import layers

inputs = backbone.input
x = backbone(inputs)

# Freeze the backbone layers
backbone.trainable = False

# This is isolating a specific embedding from RoBERTa's backbone output that is akin to the document embedding.
# This embedding is associated with the CLS (classification) token.
# The CLS token is a specific token (a fixed value) that is added at the beginning of every textual sequence in RoBERTa's training data.
# The Neural Net learns to shift this token's embedding around depending on all the words that appear in a given sentence, to improve masked word prediction.
# As the model achieves its self-supervised prediction goal, it learns how to produce a CLS token embedding for a given sequence of text that captures relevant information about the entire sentence.
# Note that this is generally more useful / less noisy than doing something like averaging all the word embeddings from the sentence.
x = x[:, 0, :]

x = layers.Dropout(0.1)(x)

# Each embedding is a 768 dimensional vector; we are allowing some transformation of the embedding with a relu activation, then doing dropout and going to a sigmoid prediction.
x = layers.Dense(768, activation="relu")(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

classifier = keras.Model(inputs, outputs)

classifier.compile(
    optimizer=keras.optimizers.Adam(5e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

classifier.fit(
    preprocessed_train_ds,
    validation_data=preprocessed_val_ds,
    epochs=5
)

Epoch 1/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 106s 59ms/step - accuracy: 0.5959 - loss: 0.6792 - val_accuracy: 0.8016 - val_loss: 0.5901
Epoch 2/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 67s 53ms/step - accuracy: 0.7809 - loss: 0.5739 - val_accuracy: 0.8570 - val_loss: 0.4998
Epoch 3/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 67s 52ms/step - accuracy: 0.8138 - loss: 0.5070 - val_accuracy: 0.8590 - val_loss: 0.4397
Epoch 4/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 67s 53ms/step - accuracy: 0.8201 - loss: 0.4634 - val_accuracy: 0.8648 - val_loss: 0.3955
Epoch 5/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 68s 53ms/step - accuracy: 0.8310 - loss: 0.4349 - val_accuracy: 0.8706 - val_loss: 0.3658
